# WORKSPACE FOR GETTING THIS INTO PURE PYTHON
 ### Zoteroize and Obsidianize a Perplexity Dialogue



 In a Perplexity dialogue copied to the clipboard by the perplexity copy button and then saved to a file, replace

 the citation numbers with matching Obsidian literature note or Zotero item links

In [1]:
import re
import pathlib as pl
import sys
from collections import defaultdict
import numpy as np
import pandas as pd
from icecream import ic
from typing import Dict, Tuple

refwrangle_dir = pl.Path('~/ref/refwrangle').expanduser() # can't reliably get dir of an .ipynb 
sys.path.append(str(refwrangle_dir))
import refwrangle as rfw
import link_perplexity_zotero as lpz

%load_ext autoreload
%autoreload 2

Reading from cache.


In [ ]:
relinker = lpz.relinker

def split_single_prs_text_perplex(pr_text: str) -> Tuple[str, str, str, str]:
    """Splits stock perplexity export markdown text into prompt, response and source sections,
    returning the source information in citenum_url_pairs.  This is the only way
    to associate citenums to urls, as stock perplexity response citenum are markdown footnotes,
    with plain citenums, like this: [citenum] or [^citenum].  These are left plain, as is."""

    match = re.search(r'(?m)^# (?P<heading_text>.+)', pr_text)
    if (heading_start_index := match.start('heading_text')) == -1:
        raise ValueError('Could not find prompt heading')
    
    preamble = pr_text[:heading_start_index].strip()
    
    prompt_end_index, response_start_index = rfw.find_markdown_divider_boundaries(pr_text)
        
    if heading_start_index >= prompt_end_index:
        raise ValueError(f'{heading_start_index=} >= {prompt_end_index=}. '
                         'Probably missed the starting level 1 header part of the prompt.')
    
    prompt = pr_text[heading_start_index:prompt_end_index+1].strip()

    response_sources_divider = '<div style="text-align: center">⁂</div>'
    response_sources_divider_index = pr_text.rfind(response_sources_divider)

    if response_sources_divider_index == -1:
        raise ValueError('Could not find divider between AI response and sources list')

    if response_sources_divider_index <= response_start_index:
        raise ValueError('body_sources_divider_index <= response_sources_divider_index')
    
    response = f"{pr_text[response_start_index:response_sources_divider_index]}".strip()
    
    sources = pr_text[response_sources_divider_index:]
    source_list_pattern_perplex = re.compile(r'\[\^?(?P<num>\d+)\]:\s*(?P<url>http[s]?://\S+)')
    citenum_url_pairs = rfw.get_link_tu_pairs(sources, source_list_pattern_perplex)
    
    return lpz.PromptResponseSplit(preamble, prompt, response, citenum_url_pairs, None) # no source titles

In [3]:
def relink_single_file_perplexity(perplexity_file: pl.Path, relinked_file: pl.Path, verbose: bool = False) -> None:
    "Relinks and writes to a file a single prompt/response from perplexity."
    file_text = rfw.read_markdown_file(perplexity_file)
    prsplit = relinker.split_single_prs_dedup(file_text, split_single_prs_text_perplex)
        
    body_relinked, relinked_sources = relinker.relink_response_and_sources(prsplit)
    body_relinked = rfw.hierarch_shift_markdown_headers(body_relinked, top_level=2)
    source_link = rfw.file_link_md('source', perplexity_file)

    relinked_file.write_text(f'{lpz.make_obsidian_front_matter()}\n*{source_link}*\n# Prompt\n\n{prsplit.prompt}\n'
                             f'# Response\n\n{body_relinked}\n# Citations\n{"\n".join(relinked_sources)}', 
                             encoding='utf-8')

In [4]:
#perplexity_dialog_file = rfw.refwrangle_test_dir / "dat" / 'perplexity_example.md'
#perplexity_dialog_file = pl.Path(r'C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/PerPlexPro.md') # >1 for one URL
# perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / "perple_new_format_longprompt_example.md"
perplexity_dialog_file = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex' / 'GPT-4o.md'

output_dir = pl.Path(r"C:\Users\scott\OneDrive\share\ref\obsidian\Obsidian Share Vault\Scratch Space")

output_file = output_dir / "tmp_perplex_example.md"
print(f'{perplexity_dialog_file=}\n-->\n{output_file=}')
verbose = False
relink_single_file_perplexity(perplexity_dialog_file, output_file, verbose)
print('Done.')

perplexity_dialog_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/refwrangle/test/dat/merge_chats_perplex/GPT-4o.md')
-->
output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/tmp_perplex_example.md')
Done.


### Test merging

In [5]:
tmpdir = rfw.refwrangle_test_dir / 'tmp'
tmpdir.mkdir(parents=True, exist_ok=True)

datdir = rfw.refwrangle_test_dir / 'dat' / 'merge_chats_perplex'
datdir.mkdir(parents=True, exist_ok=True)

# multi-file perplex, same prompt
#chat_files = list(datdir.glob('*.md'))
# multi-file, different prompt
#chat_files = [chat_files[3], pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_example.md")]
# single file
#chat_files = [chat_files[3]]

# single smc file but multiprompt
chat_files = [pl.Path(r"C:\Users\scott\OneDrive\share\ref\refwrangle\test\dat\perplexity_multi_prompt_savemychatbot_example.md")]


merged_output_file = output_dir / 'tmp_perplexy_merged.md'

##### Fix any duplicate cite numbers inside of each body and collect them

In [ ]:

# def get_prompt_response(chat_fle):
#     file_text = rfw.read_markdown_file(chat_file)

#     if lpz.is_smc_content(file_text):
#         prs_splits = []    
#         sections = re.split(rf'(?<=\n){lpz.PROMPT_HEADER_SMC}', file_text)
#         for section in sections[1:]:  # Process each user section
#             section = f'{lpz.PROMPT_HEADER_SMC}\n{section}' # stick header back on for more certtain matching
#             dedup_prs = relinker.split_single_prs_dedup(section, lpz.split_single_prs_text_smc)
#             prs_splits.append(dedup_prs)

#         return prs_splits

#     
#     return [relinker.split_single_prs_dedup(file_text, split_single_prs_text_perplex)]



def load_and_dedup_perplexity_files(chat_files, verbose = True):
    num_chat_files = len(chat_files)
    is_multi_prompt_file = []
    all_prompts, all_responses, all_citenums_to_url = [], [], []
    
    for file_index, chat_file in enumerate(chat_files):
        if verbose:
            print(f'Parsing {chat_file.name}')
            
        file_text = rfw.read_markdown_file(chat_file)

        # split the file into prompt, response and source section (if SMC)
        if lpz.is_smc_content(file_text):
            # SMC files can have multiple prompts and responses
            prs_splits = []    
            sections = re.split(rf'(?<=\n){lpz.PROMPT_HEADER_SMC}', file_text)
            for section in sections[1:]:  # Process each user section
                section = f'{lpz.PROMPT_HEADER_SMC}\n{section}' # stick header back on for more certtain matching
                dedup_prs = relinker.split_single_prs_dedup(section, lpz.split_single_prs_text_smc)
                prs_splits.append(dedup_prs)
        else:
            # stock perplexity files have only a single prompt-response pair
            prs_splits = [relinker.split_single_prs_dedup(file_text, split_single_prs_text_perplex)]

        is_multi_prompt_file.append(len(prs_splits) > 1)
        
        for prompt_index, prsplit in enumerate(prs_splits):
            all_prompts.append(prsplit.prompt)
            all_responses.append(prsplit.response_dedup)

            citenum_to_url_df = prsplit.citenum_to_url_df.copy().reset_index()
            citenum_to_url_df[['file_index','chat_file', 'prompt_index']] = file_index, chat_file, prompt_index
            if prsplit.url_to_source_title is not None:
                citenum_to_url_df = citenum_to_url_df.set_index('url', drop=True)
                citenum_to_url_df['title'] = prsplit.url_to_source_title            
                citenum_to_url_df = citenum_to_url_df.reset_index()

            all_citenums_to_url.append(citenum_to_url_df)
        
    all_citenums_to_url = pd.concat(all_citenums_to_url)

    if 'title' in all_citenums_to_url.columns:
        # Fill in titles when find them in other smc prompt-response pairs (useful for debugging?)"""
        fixed_title_dfs = []
        for url, df in all_citenums_to_url.groupby('url'):
            has_no_title = df.title.isna()

            if any(has_no_title):
                if len(titles := df.title[~has_no_title].unique()) > 1:
                    ic(url, titles)
                    raise ValueError('Different titles for same URL')

                if len(titles) > 0:
                    df = df.fillna({'title': titles[0]})

            fixed_title_dfs.append(df.copy())
        all_citenums_to_url = pd.concat(fixed_title_dfs)

        # fix at the end to allow possible title fill-in from other responses
        all_citenums_to_url['title'] = all_citenums_to_url.title.fillna('NO TITLE: likely bare citenum in response w/ no URL')
    
    return num_chat_files, is_multi_prompt_file, all_prompts, all_responses, all_citenums_to_url

Parsing perplexity_multi_prompt_savemychatbot_example.md
Sources header(\*\*Sources:\*\*) not in expected place or no source list: Assume no sources.
Malformed Plain citenum [20] appears without URL in response
num_url_pair[0]='1', num_url_pair[1]='https://libanswers.lib.miamioh.edu/stats-faq/faq/343635' in response but not source list
num_url_pair[0]='3', num_url_pair[1]='https://stats.stackexchange.com/questions/81659/mutual-information-versus-correlation' in response but not source list
num_url_pair[0]='17', num_url_pair[1]='https://towardsdatascience.com/how-to-measure-relationship-between-variables-d0606df27fd8' in response but not source list
num_url_pair[0]='7', num_url_pair[1]='https://mattiheino.com/2019/05/10/correlation' in response but not source list
num_url_pair[0]='10', num_url_pair[1]='https://m-clark.github.io/docs/correlationcomparison.pdf' in response but not source list
num_url_pair[0]='9', num_url_pair[1]='https://www.stats.ox.ac.uk/~cucuring/lecture_2_correlations

In [ ]:
# def fill_missing_titles_smc(all_citenums_to_url: pd.DataFrame) -> pd.DataFrame:
#     """Fill in titles when find them in other smc prompt-response pairs (useful for debugging?)"""
#     fixed_title_dfs = []
#     for url, df in all_citenums_to_url.groupby('url'):
#         has_no_title = df.title.isna()

#         if any(has_no_title):
#             if len(titles := df.title[~has_no_title].unique()) > 1:
#                 ic(url, titles)
#                 raise ValueError('Different titles for same URL')

#             if len(titles) > 0:
#                 df = df.fillna({'title': titles[0]})

#         fixed_title_dfs.append(df.copy())
#     all_citenums_to_url = pd.concat(fixed_title_dfs)

#     # fix at the end to allow possible title fill-in from other responses
#     all_citenums_to_url['title'] = all_citenums_to_url.title.fillna('NO TITLE: likely bare citenum in response w/ no URL')
#     return all_citenums_to_url


# if 'title' in all_citenums_to_url.columns:
#      all_citenums_to_url = fill_missing_titles_smc(all_citenums_to_url)

#### Make a unified cite number set for the merged document

In [ ]:
def reorder_merged_citenums(all_citenums_to_url: pd.DataFrame) -> pd.DataFrame:
    """ Reorder the merged citenums, giving each url a new, unique citenum. 
    Urls get lower new citenums when they're mostly in early prompts of early files and with
    mostly low original citenums."""
    df = all_citenums_to_url.copy()
    df['dedup_num_int'] = df['dedup_num'].astype(int)  # so can sort

    url_ranks = df.groupby('url').agg(
        mean_file_index=('file_index', 'mean'),
        mean_dedup_num_int=('dedup_num_int', 'mean'),
        mean_prompt_index=('prompt_index', 'mean')
    ).reset_index()

    url_ranks = url_ranks.sort_values(by=['mean_file_index', 'mean_prompt_index', 'mean_dedup_num_int'], 
                                      ascending=True).reset_index(drop=True)

    url_ranks['unif_num'] = np.arange(1, len(url_ranks) + 1).astype(str)  # citenum == rank as string

    # Merge back the new citenums
    df = df.merge(url_ranks[['url', 'unif_num']], on='url')
    all_citenums_to_url = (df.sort_values(by='unif_num', key=lambda col: col.astype(int))
                           .rename(dict(new_num='doc_dedup_num', citenum_merged='new_num'), axis=1)
                           .drop('dedup_num_int', axis=1)
                           .set_index('file_index'))
    
    return all_citenums_to_url

#### Assign new, unified cite numbers to each body and concatenate them into a single string

In [ ]:
# Put prompts and headers within headings

def concat_prompts_responses(all_prompts, all_responses, chat_files, all_citenums_to_url, relinker, num_chat_files, is_multi_prompt_file):
    all_prompts_same = True
    for i in range(0, len(all_prompts) - 1):
        is_same = all_prompts[i].strip().lower() == all_prompts[i + 1].strip().lower()
        all_prompts_same &= is_same

    full_prompt_indices = rfw.unique_rows(all_citenums_to_url.reset_index(), ['file_index', 'prompt_index'])
    all_citenums_to_url = (all_citenums_to_url
                           .reset_index()
                           .set_index(['file_index', 'prompt_index'])
                           .sort_index())

    all_promptresp, chat_source_file_link = '', []
    for global_index, (file_index, prompt_index) in full_prompt_indices.iterrows():

        # remap deduped citenums to unified citenums
        citenums_to_url_this = all_citenums_to_url.loc[file_index, prompt_index].copy()
        citenums_dedup_to_unified = citenums_to_url_this.set_index('dedup_num').unif_num.to_dict()
        response_unified = relinker.replace_response_citenums(all_responses[global_index], citenums_dedup_to_unified)  # unified citenums

        def header_add(name, level):
            nonlocal all_promptresp
            all_promptresp += f'{rfw.make_atx_header(name, level)}\n'

        def text_add(text, top_level):
            nonlocal all_promptresp
            text = rfw.hierarch_shift_markdown_headers(text, top_level).strip()
            all_promptresp += f'{text}\n'

        def prompt_add(top_level):
            text_add(all_prompts[global_index], top_level)

        def header_short_prompt_add(level):
            name = rfw.get_first_n_words(all_prompts[global_index], n=10)
            header_add(name, level)

        def header_short_filename_add(level):
            header_add(chat_files[file_index].name, level)

        def response_add(top_level):
            text_add(response_unified, top_level)

        def file_link_add():
            nonlocal all_promptresp
            this_file = chat_files[file_index]
            all_promptresp += f'{rfw.file_link_md('source', this_file)}\n'

        # Put this prompt-response within the appropriate merged dialog headings
        if num_chat_files == 1:
            file_link_add()
            if prompt_index == 0:
                if is_multi_prompt_file[file_index]:
                    header_short_prompt_add(1)
                else:
                    header_add("Prompt", 1)
                prompt_add(3)
                header_add("Response", 2)
                response_add(3)
            else:
                header_short_prompt_add(1)
                prompt_add(3)
                header_add("Response", 2)
                response_add(3)
        else:
            if all_prompts_same and not is_multi_prompt_file[file_index]:
                # Print prompt once at top, then responses
                if file_index == 0 and prompt_index == 0:
                    header_add("Prompt", 1)
                    prompt_add(2)
                    header_add("Responses", 1)

                header_short_filename_add(2)
                file_link_add()
                response_add(3)
            else:
                # print all prompt/response w/ separate prompt/resp headers
                if prompt_index == 0:
                    header_short_filename_add(1)
                    file_link_add()

                header_add("Prompt", 2)
                prompt_add(3)
                header_add("Response", 2)
                response_add(3)

    return all_promptresp, chat_source_file_link



##### Insert links to Obsidian or Zotero

In [ ]:
def relink_responses_sources_and_save(all_citenums_to_url, all_promptresp, relinker, merged_output_file):
    url_to_source_title = {}
    if 'title' in all_citenums_to_url.columns:
        for unif_num, df in all_citenums_to_url.reset_index().groupby('unif_num'):
            url_to_source_title[df.iloc[0].url] = df.iloc[0].title

    prsplit_all = lpz.PromptResponseSplitDeDup('', '', all_promptresp, all_citenums_to_url, url_to_source_title)

    all_promptresp_unified_relinked, relinked_sources = relinker.relink_response_and_sources(prsplit_all, 'unif_num')
    relinked_sources = "\n".join(sorted(relinked_sources, key=lambda line: int(re.search(lpz.citenum_plain_re, line).group('num'))))

    print(f'writing to {merged_output_file=}')
    merged_output_file.write_text(f'{lpz.make_obsidian_front_matter()}\n{all_promptresp_unified_relinked}\n# Citations\n{relinked_sources}',
                                  encoding='utf-8')
    print("Done.")

writing to merged_output_file=WindowsPath('C:/Users/scott/OneDrive/share/ref/obsidian/Obsidian Share Vault/Scratch Space/tmp_perplexy_merged.md')
Done.


In [ ]:
def relink_chats(chat_files, verbose=True):
    
    num_chat_files, is_multi_prompt_file, all_prompts, all_responses, all_citenums_to_url = load_and_dedup_perplexity_files(chat_files)

    all_citenums_to_url = reorder_merged_citenums(all_citenums_to_url)

    (all_promptresp,
    chat_source_file_link) = concat_prompts_responses(all_prompts, all_responses, chat_files, 
                                                        all_citenums_to_url, relinker, num_chat_files, 
                                                        is_multi_prompt_file)
    
    relink_responses_sources_and_save(all_citenums_to_url, all_promptresp, relinker, merged_output_file)